# 🥣 Kohlenhydrat-Schätzer (Together.ai-Variante)

Adaption des ursprünglichen Notebooks anthropic-estimator.ipynb auf die [Together.ai](https://together.ai)-API.

## Modellwahl: `Qwen/Qwen3-235B-A22B-Instruct-2507-tput`

Aus dem aktuellen [Serverless-Katalog](https://docs.together.ai/docs/serverless-models) ausgewählt, weil:

| Kriterium | Warum es hier zählt |
|---|---|
| Mehrsprachigkeit (auch Deutsch) | Wir parsen Wendungen wie *"ein Viertel Apfel"* |
| Native **Structured Outputs** | JSON-Schema wird vom Modell erzwungen, nicht erbeten |
| 235B Parameter | Solides Weltwissen über Lebensmittel |
| $0.20 / $1M Input — $0.60 / $1M Output | Sehr günstig für diese Größenklasse |
| Serverless | Kein Endpoint-Setup nötig |

**Alternativen** je nach Vorliebe: `google/gemma-4-31B-it` (kleiner, ähnlich gut in Deutsch) oder `openai/gpt-oss-120b` (noch billiger).

## Architektur (DDD, unverändert)

Wie ein Restaurant-Team — wir tauschen nur das Nachschlagewerk:

| DDD-Begriff | Hier konkret | Restaurant-Analogie |
|---|---|---|
| Value Object | `Carbohydrates`, `Ingredient` | Posten auf der Rechnung |
| Aggregate Root | `Meal` | Die ganze Bestellung |
| Domain Service | `MealAnalyzer` | Der Ernährungsberater |
| Infrastructure | **Together.ai-API mit Qwen3** | Sein neues Nachschlagewerk |

Das Domain-Modell bleibt **byte-identisch** zur Anthropic-Variante. Genau dafür ist DDD da.

## 1. Setup

In [ ]:
# %pip install together

In [1]:
import os
import json
from dotenv import load_dotenv
from dataclasses import dataclass, field
from typing import List
from together import Together

# API-Key setzen (entweder hier oder als Umgebungsvariable TOGETHER_API_KEY)
load_dotenv()
# os.environ['TOGETHER_API_KEY'] = '...'

client = Together()  # liest automatisch TOGETHER_API_KEY

## 2. Domain Model

Reine Datenstrukturen, keine API-Abhängigkeit. Identisch zur Anthropic-Variante.

In [2]:
@dataclass(frozen=True)
class Carbohydrates:
    """Wertobjekt: Kohlenhydratmenge in Gramm. Unveränderlich, addierbar."""
    grams: float

    def __add__(self, other: "Carbohydrates") -> "Carbohydrates":
        return Carbohydrates(self.grams + other.grams)

    def __radd__(self, other):
        if other == 0:
            return self
        return self.__add__(other)

    def __str__(self) -> str:
        return f"{self.grams:.1f} g"


@dataclass(frozen=True)
class Ingredient:
    """Wertobjekt: eine einzelne Zutat mit Schätzungen."""
    name: str
    estimated_weight_g: float
    carbs: Carbohydrates


@dataclass
class Meal:
    """Aggregate Root: eine Mahlzeit besteht aus Zutaten."""
    description: str
    ingredients: List[Ingredient] = field(default_factory=list)

    @property
    def total_carbs(self) -> Carbohydrates:
        return sum(i.carbs for i in self.ingredients) or Carbohydrates(0)

    def report(self) -> str:
        lines = [f"Mahlzeit: {self.description}", "-" * 50]
        for ing in self.ingredients:
            lines.append(
                f"  • {ing.name:<30} "
                f"{ing.estimated_weight_g:>6.1f} g  →  {ing.carbs}"
            )
        lines.append("-" * 50)
        lines.append(f"  Gesamt-Kohlenhydrate: {self.total_carbs}")
        return "\n".join(lines)

## 3. JSON-Schema für strukturierte Antworten

Together unterstützt für Qwen3 die `response_format`-Option mit JSON-Schema. Das Modell darf dann **nur** Output produzieren, der dieses Schema erfüllt — kein Fallback-Parsing nötig.

*Analogie: statt den Kellner zu bitten, „bitte ordentlich aufschreiben", drücken wir ihm ein Formular in die Hand. Er **kann** gar nicht mehr formlos antworten.*

In [3]:
MAHLZEIT_SCHEMA = {
    "type": "object",
    "properties": {
        "zutaten": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "Name der Zutat auf Deutsch",
                    },
                    "gewicht_g": {
                        "type": "number",
                        "description": "Geschätztes Gewicht in Gramm",
                    },
                    "kohlenhydrate_g": {
                        "type": "number",
                        "description": "Geschätzte Kohlenhydrate in Gramm",
                    },
                },
                "required": ["name", "gewicht_g", "kohlenhydrate_g"],
            },
        }
    },
    "required": ["zutaten"],
}

## 4. Domain Service: `MealAnalyzer` (Together-Variante)
- below define the together model to be used 
- note that it has to support serverless deployment, which is frequently phased out

In [16]:
#together_model = "arize-ai/qwen-2-1.5b-instruct" 
together_model = "google/gemma-3n-E4B-it"   # "deepseek-ai/DeepSeek-V4-Flash-0731" 

In [17]:
SYSTEM_PROMPT = """Du bist ein Ernährungsexperte. Du erhältst eine umgangssprachliche \
Beschreibung einer Mahlzeit auf Deutsch. Gib für JEDE Zutat zurück:
  - name (string, deutsch)
  - gewicht_g (float, geschätztes Gewicht in Gramm)
  - kohlenhydrate_g (float, Kohlenhydrate in Gramm)

Faustregeln für übliche Mengen:
  - 1 Suppenlöffel (Esslöffel) Trockenes (Haferflocken, Mehl) ≈ 10–15 g
  - 1 Teelöffel ≈ 5 g
  - 1 Tasse ≈ 200–250 ml
  - mittelgroßer Apfel ≈ 180 g, Banane ≈ 120 g, Pfirsich ≈ 150 g
  - 1 Scheibe Brot ≈ 30 g

Antworte NUR mit dem geforderten JSON-Objekt — keine Erklärungen."""


class MealAnalyzer:
    """Domain Service: übersetzt freien Text in ein strukturiertes Meal über Together.ai."""

    def __init__(
        self,
        client: Together,
        model: str = together_model,
    ):
        self.client = client
        self.model = model

    def analyze(self, description: str) -> Meal:
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": description},
            ],
            response_format={
                "type": "json_object",
                "schema": MAHLZEIT_SCHEMA,
            },
            temperature=0.2,  # niedrig: wir wollen konsistente Schätzungen
            max_tokens=1024,
        )

        raw = response.choices[0].message.content
        data = json.loads(raw)  # garantiert valides JSON dank Schema

        ingredients = [
            Ingredient(
                name=z["name"],
                estimated_weight_g=float(z["gewicht_g"]),
                carbs=Carbohydrates(grams=float(z["kohlenhydrate_g"])),
            )
            for z in data["zutaten"]
        ]
        return Meal(description=description, ingredients=ingredients)

## 5. Verwendung

Dein Beispiel:

In [9]:
analyzer = MealAnalyzer(client)

#beschreibung = "2 Suppenlöffel Haferflocken, ein Viertel Apfel, eine halbe Banane, ein halber Pfirsich"
#beschreibung = "halbe Birne, ein viertel Apfel, 3 Erdbeere, 10 g getrocknete Mango"
#beschreibung = "42 g Apfel, 55 g Banane, 56 g Erdbeeren, 65 g Pfirsich"
beschreibung = "96 g Feige, 116 g Kiwi"

mahlzeit = analyzer.analyze(beschreibung)
print(mahlzeit.report())

Mahlzeit: 96 g Feige, 116 g Kiwi
--------------------------------------------------
  • Feige                            96.0 g  →  15.0 g
  • Kiwi                            116.0 g  →  15.0 g
--------------------------------------------------
  Gesamt-Kohlenhydrate: 30.0 g


### Nur die Zahl ausgeben

In [ ]:
print(f"Kohlenhydrate: {mahlzeit.total_carbs.grams:.1f} g")

## 6. Weitere Beispiele zum Ausprobieren

In [20]:
beispiele = [
    "eine Brezel mit Butter",
    "70 g Spaghetti mit Tomatensoße",
    "96 g Feige und 116 g Kiwi",
    "80 g Mandelkuchen",
]

for b in beispiele:
    m = analyzer.analyze(b)
    print(m.report())
    print()

Mahlzeit: eine Brezel mit Butter
--------------------------------------------------
  • Brot                             30.0 g  →  20.0 g
  • Butter                            5.0 g  →  0.0 g
--------------------------------------------------
  Gesamt-Kohlenhydrate: 20.0 g

Mahlzeit: 70 g Spaghetti mit Tomatensoße
--------------------------------------------------
  • Spaghetti                        70.0 g  →  10.0 g
  • Tomatensoße                      10.0 g  →  10.0 g
--------------------------------------------------
  Gesamt-Kohlenhydrate: 20.0 g

Mahlzeit: 96 g Feige und 116 g Kiwi
--------------------------------------------------
  • Feige                            96.0 g  →  14.0 g
  • Kiwi                            116.0 g  →  15.0 g
--------------------------------------------------
  Gesamt-Kohlenhydrate: 29.0 g

Mahlzeit: 80 g Mandelkuchen
--------------------------------------------------
  • Mandelkuchen                     80.0 g  →  0.0 g
--------------------------

## 7. Modellvergleich (optional)

Da der `MealAnalyzer` das Modell als Parameter akzeptiert, kannst du verschiedene Modelle gegeneinander testen:

In [ ]:
kandidaten = [
    "arize-ai/qwen-2-1.5b-instruct",
    "deepseek-ai/DeepSeek-V4-Flash-0731",
    "openai/gpt-oss-20b",
]

for model in kandidaten:
    print(f"--- {model} ---")
    a = MealAnalyzer(client, model=model)
    try:
        for b in beispiele:
            m = a.analyze(b)
            print(m.report())
            print()
    except Exception as e:
        print(f"Error for {model}: {e}\n")

--- arize-ai/qwen-2-1.5b-instruct ---
Mahlzeit: eine Brezel mit Butter
--------------------------------------------------
  • Brezel                          150.0 g  →  50.0 g
  • Butter                           10.0 g  →  0.0 g
--------------------------------------------------
  Gesamt-Kohlenhydrate: 50.0 g

Mahlzeit: 70 g Spaghetti mit Tomatensoße
--------------------------------------------------
  • Spaghetti                        70.0 g  →  10.0 g
  • Tomatensoße                      10.0 g  →  10.0 g
--------------------------------------------------
  Gesamt-Kohlenhydrate: 20.0 g

Mahlzeit: 96 g Feige und 116 g Kiwi
--------------------------------------------------
  • Feige                            96.0 g  →  12.0 g
  • Kiwi                            116.0 g  →  12.0 g
--------------------------------------------------
  Gesamt-Kohlenhydrate: 24.0 g

Mahlzeit: 80 g Mandelkuchen
--------------------------------------------------
  • Mandelkuchen                     80.0 

## 8. Erweiterungsideen

- **Eiweiß und Fett** zusätzlich schätzen → neue Felder im Schema, neues Value Object, gleiche Architektur.
- **Lokale Datenbank** (BLS, USDA) als alternative Infrastructure → man könnte ein `Protocol`/Interface für `MealAnalyzer` definieren und je nach Bedarf API oder lokale DB injecten. Domain bleibt unverändert. Genau dafür existiert die DDD-Schichtung.
- **Plausibilitäts-Validierung** im Domain Model (z. B. `kohlenhydrate_g <= gewicht_g`).
- **Streaming** mit `stream=True` für lange Mahlzeiten.

## ⚠️ Hinweis

Die Werte sind **LLM-Schätzungen**, keine kalibrierten Labormessungen. Für medizinische Zwecke (Diabetes-Berechnungen etc.) bitte mit offiziellen Nährwerttabellen wie der BLS gegenprüfen.